In [ ]:
from libraries import *
from parameters import *
import statsmodels.api as sm
import sklearn.feature_selection
import statsmodels.api as sm
import sklearn.feature_selection
from sklearn.preprocessing import PolynomialFeatures
import warnings
warnings.filterwarnings('ignore')
from numpy.random import seed
from numpy.random import randint
import statsmodels.formula.api as smf

import statsmodels.api as sm
%load_ext rpy2.ipython

import sys

In [ ]:
os.getcwd()
os.chdir(projectDir)

In [ ]:
%load_ext rpy2.ipython

In [ ]:
adata = sc.read(ANNDATA_ALL_KO)

In [ ]:
adata = adata[adata.obs["Doubles"] == 0,:]

In [ ]:
adata.obs["KOModule"] = "K_CONTROL"
adata.obs.loc[adata.obs["K_0"] == 1,"KOModule"] = "K_0"
adata.obs.loc[adata.obs["K_1"] == 1,"KOModule"] = "K_1"
adata.obs.loc[adata.obs["K_2"] == 1,"KOModule"] = "K_2"
adata.obs.loc[adata.obs["K_3"] == 1,"KOModule"] = "K_3"
adata.obs.loc[adata.obs["K_4"] == 1,"KOModule"] = "K_4"
adata.obs.loc[adata.obs["K_5"] == 1,"KOModule"] = "K_5"

In [ ]:
allVars = pd.Series(["K_0","K_1","K_2","K_3","K_4","K_5"]) 
myForm = allVars

In [ ]:
myForm = myForm.append(pd.Series(["n_genes","mt_frac","leiden"]))

myFormula = "+".join(myForm)
my_formula = "y~" + myFormula
my_formula

In [ ]:
allExpMat = pd.DataFrame(adata.X)
allExpMat.columns = adata.var_names
allResp = allExpMat.columns

In [ ]:
guideMatrix = adata.obs[["K_0","K_1","K_2","K_3","K_4","K_5"]]

df = adata.obs[["n_genes", "mt_frac", "leiden" ]]

guideMatrix = guideMatrix.join(df)
guideMatrix

In [ ]:
%%R -i guideMatrix,allExpMat,my_formula  -w 8 -h 4 -u in
library("MASS")
library("broom")

coefDF = data.frame()

for(i in seq(1,ncol(allExpMat),1)){
      print(i)
    
          
      tryCatch(
            expr = {
                   guideMatrix["y"] = allExpMat[,i]
                   myFit <- glm(formula(my_formula), data = guideMatrix, family = gaussian())
                   myDF <- data.frame(tidy(myFit))
                   myDF$respGene = colnames(allExpMat)[i]
                   coefDF <- rbind(coefDF, myDF)
            },
            error = function(e){ 
                print(e)
            },
            finally = {
                # (Optional)
                # Do this at the end before quitting the tryCatch structure...
            }
      ) 

}


write.csv(coefDF, paste0("outputs/KOModuleEffects.csv"), quote=FALSE)